# Two-dimensional rectangular particle in a box

**Prerequisite:** [Two-dimensional rectangular particle-in-a-box spectrum](../../../../../docs/lecture-notes/qm/piab2d/tise/analytical/piab2d.md)

This maintained laboratory evaluates the continuum stationary-state energies of a particle confined by homogeneous Dirichlet boundaries on a rectangle. The selected examples are nondimensional, with $m=1$ and $\hbar=1$.

The predecessor notebook mixed this Dirichlet model with a periodic reciprocal-space calculation. This laboratory retains only the rectangular particle-in-a-box model; periodic free-particle calculations require a separate physical and boundary-condition contract.

## Learning objectives

1. Enumerate states by explicit positive quantum-number pairs $(n_x,n_y)$.
2. Evaluate and independently check the analytical spectrum.
3. Identify exchange degeneracy in a square box.
4. Observe how unequal side lengths break that exchange degeneracy.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.qm.piab2d.base import Piab2D
from projectkoios.physkit.qm.piab2d.tise.analytical.solution import (
    Piab2DAnalyticalEvaluator,
)
from projectkoios.physkit.units import UnitSystem

In [ ]:
unit_box_length = 1.0
unit_particle_mass = 1.0
square_model = Piab2D(
    length_x=unit_box_length,
    length_y=unit_box_length,
    mass=unit_particle_mass,
    unit_system=UnitSystem.NONDIMENSIONAL,
)
maximum_quantum_number = 4
square_solution = Piab2DAnalyticalEvaluator().execute(
    model=square_model,
    maximum_quantum_number_x=maximum_quantum_number,
    maximum_quantum_number_y=maximum_quantum_number,
)

square_solution.quantum_numbers[:10], square_solution.energies.magnitude[:10]

In [ ]:
quantum_numbers = square_solution.quantum_numbers.astype(np.float64)
expected_energies = (
    np.pi**2
    / (2.0 * square_model.mass)
    * (
        quantum_numbers[:, 0] ** 2 / square_model.length_x**2
        + quantum_numbers[:, 1] ** 2 / square_model.length_y**2
    )
)

assert np.allclose(
    square_solution.energies.magnitude,
    expected_energies,
    rtol=2e-15,
    atol=0.0,
)
assert np.all(np.diff(square_solution.energies.magnitude) >= 0.0)

In [ ]:
integer_mode_sums = np.sum(square_solution.quantum_numbers**2, axis=1)
unique_mode_sums, multiplicities = np.unique(
    integer_mode_sums,
    return_counts=True,
)
degenerate_levels = np.column_stack(
    (unique_mode_sums[multiplicities > 1], multiplicities[multiplicities > 1])
)

pair_to_energy = {
    tuple(pair): energy
    for pair, energy in zip(
        square_solution.quantum_numbers,
        square_solution.energies.magnitude,
        strict=True,
    )
}
assert pair_to_energy[(1, 2)] == pair_to_energy[(2, 1)]
assert pair_to_energy[(1, 3)] == pair_to_energy[(3, 1)]

degenerate_levels

In [ ]:
rectangular_length_x = 1.0
rectangular_length_y = 1.5
rectangular_maximum_quantum_number = 3
rectangular_model = Piab2D(
    length_x=rectangular_length_x,
    length_y=rectangular_length_y,
    mass=unit_particle_mass,
    unit_system=UnitSystem.NONDIMENSIONAL,
)
rectangular_solution = Piab2DAnalyticalEvaluator().execute(
    model=rectangular_model,
    maximum_quantum_number_x=rectangular_maximum_quantum_number,
    maximum_quantum_number_y=rectangular_maximum_quantum_number,
)
rectangular_pair_to_energy = {
    tuple(pair): energy
    for pair, energy in zip(
        rectangular_solution.quantum_numbers,
        rectangular_solution.energies.magnitude,
        strict=True,
    )
}

assert not np.isclose(
    rectangular_pair_to_energy[(1, 2)],
    rectangular_pair_to_energy[(2, 1)],
)
(
    rectangular_pair_to_energy[(1, 2)],
    rectangular_pair_to_energy[(2, 1)],
)

In [ ]:
state_indices = np.arange(
    1,
    square_solution.energies.magnitude.size + 1,
    dtype=np.int64,
)
labels = [rf"$({n_x},{n_y})$" for n_x, n_y in square_solution.quantum_numbers]

figure, axis = plt.subplots(figsize=(10, 5))
axis.scatter(state_indices, square_solution.energies.magnitude)
for state_index, energy, label in zip(
    state_indices[:10],
    square_solution.energies.magnitude[:10],
    labels[:10],
    strict=True,
):
    axis.annotate(
        label, (state_index, energy), xytext=(3, 5), textcoords="offset points"
    )
axis.set(
    xlabel="energy-ordered state index",
    ylabel="energy",
    title="First 16 states of the nondimensional square box",
)
axis.grid(True)
figure.tight_layout()
plt.show()

## Interpretation

The evaluator orders the finite state inventory by energy and then lexicographically resolves exact ties. A square box is invariant under exchange of the coordinate axes, so $(n_x,n_y)$ and $(n_y,n_x)$ have equal energy. Unequal side lengths remove that exchange symmetry and generally split the pair.

This analytical result is a continuum Dirichlet spectrum. It is not a finite-difference approximation and not a periodic reciprocal-space spectrum.

## Exercises

1. Increase both maximum quantum numbers and identify levels with multiplicity greater than two.
2. Double both side lengths and verify inverse-square energy scaling.
3. Hold $L_x$ fixed while varying $L_y$ and track the $(1,2)$ and $(2,1)$ levels.
4. Derive and plot the normalized product eigenfunctions for selected states.
